# Predicting Hazardous Near-Earth Objects (NEOs)

**Goal:** build a machine-learning classifier that predicts whether a near-Earth object is potentially hazardous, using its size, brightness, speed and miss distance.

**Dataset:** [NASA Nearest Earth Objects (1910-2024)](https://www.kaggle.com/datasets/ivansher/nasa-nearest-earth-objects-1910-2024) from Kaggle.

**Workflow:** data loading and cleaning, exploratory analysis, train/test split, preprocessing, class balancing with SMOTE (training data only), model training (Logistic Regression and Random Forest), and evaluation.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
TARGET = "is_hazardous"

# Put the downloaded Kaggle file next to this notebook, or change the path/name here.
DATA_PATH = "nearest-earth-objects(1910-2024).csv"

## 2. Load the data

In [ ]:
if DATA_PATH.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

## 3. Inspect the data

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicated rows:", df.duplicated().sum())
df.describe().T

## 4. Clean the data

In [ ]:
df = df.drop_duplicates().copy()

# Identifier / constant columns carry no predictive signal
drop_cols = [c for c in ["neo_id", "name"] if c in df.columns]
if "orbiting_body" in df.columns and df["orbiting_body"].nunique() <= 1:
    drop_cols.append("orbiting_body")
df = df.drop(columns=drop_cols)

# Make the target a clean 0/1 column
df[TARGET] = (df[TARGET].astype(str).str.strip().str.lower()
              .map({"true": 1, "false": 0, "1": 1, "0": 0}))
df = df.dropna(subset=[TARGET])
df[TARGET] = df[TARGET].astype(int)

# Make sure every feature is numeric
feature_cols = [c for c in df.columns if c != TARGET]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

print("Dropped columns:", drop_cols)
print("Features used:", feature_cols)
print("Shape after cleaning:", df.shape)

## 5. Exploratory data analysis

In [ ]:
# Class balance: hazardous objects are usually the minority class
counts = df[TARGET].value_counts().sort_index()
print(counts)
print((counts / counts.sum()).round(3))

plt.figure(figsize=(5, 4))
sns.countplot(x=TARGET, data=df)
plt.title("Class distribution (0 = not hazardous, 1 = hazardous)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="absolute_magnitude", hue=TARGET, bins=40, kde=True, stat="density", common_norm=False)
plt.title("Absolute magnitude by class")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df.sample(min(len(df), 5000), random_state=RANDOM_STATE),
                x="relative_velocity", y="miss_distance", hue=TARGET, alpha=0.5)
plt.yscale("log")
plt.title("Relative velocity vs. miss distance (log scale)")
plt.show()

## 6. Train/test split

The split happens **before** any scaling, imputation or oversampling, so no information from the test set leaks into training. `stratify` keeps the same class ratio in both sets.

In [ ]:
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 7. Preprocessing and class balancing

- Missing values are filled with the **median** (fitted on training data only).
- Features are standardized (fitted on training data only).
- **SMOTE is applied to the training set only**; the test set stays untouched so the evaluation reflects the real class distribution.

In [ ]:
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_p = scaler.fit_transform(imputer.fit_transform(X_train))
X_test_p = scaler.transform(imputer.transform(X_test))

smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train_p, y_train)

print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE: ", np.bincount(y_train_bal))

## 8. Train the models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    print("Trained:", name)

## 9. Evaluate

Accuracy can be misleading on imbalanced data, so the models are compared with **precision, recall, F1 and ROC-AUC** on the untouched test set. For a hazard-detection task, **recall** (not missing real hazards) matters especially.

In [ ]:
rows, probas = [], {}
for name, model in models.items():
    pred = model.predict(X_test_p)
    proba = model.predict_proba(X_test_p)[:, 1]
    probas[name] = proba
    rows.append({
        "Model": name,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
results

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(11, 4))
for ax, (name, model) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test_p))
    ConfusionMatrixDisplay(cm, display_labels=["Not hazardous", "Hazardous"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
for name, proba in probas.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves")
plt.legend()
plt.show()

## 10. Which features matter most?

In [ ]:
rf = models["Random Forest"]
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values()

plt.figure(figsize=(7, 4))
importance.plot(kind="barh")
plt.title("Random Forest feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 11. Conclusions

After running the notebook, summarize here in 3 to 4 sentences:

- Which model performed best, and on which metric.
- Which features drove the predictions (see the feature importance plot).
- One limitation (for example: the minority class is small, so recall can vary between runs) and one idea for improvement (for example: hyperparameter tuning or cross-validation).